## Labels & selectors

Once you have more than a handful of pods, "list everything and eyeball it" stops scaling. **Labels** are the fix — key/value pairs in `metadata.labels`, built for *identifying* and *selecting* groups of objects.

```yaml
metadata:
  labels:
    app: web
    tier: frontend
    version: v2
```

Values are short strings; an object can carry any number. The community convention is the `app.kubernetes.io/*` set — `name`, `instance`, `version`, `component`, `part-of`, `managed-by` — which dashboards, Helm, and Argo CD understand automatically. Change them on the fly: `kubectl label pod web env=prod`, `--overwrite` to change, trailing `-` to remove, `--show-labels` to list.

A **selector** filters objects by their labels. Two flavours:

- **Equality-based** — `app=web`, `tier!=frontend`, `app=web,env=prod` (comma = AND, **no OR**).
- **Set-based** — `environment in (prod,staging)`, `tier notin (...)`, `release` (exists), `!release` (does not exist).

```bash
kubectl get pods -l app=web
kubectl get pods -l 'environment in (prod,staging)'
kubectl get pods -l '!canary'
```

Selectors are **everywhere**: `kubectl -l`, a **Service**'s `spec.selector`, a Deployment/ReplicaSet/DaemonSet/Job's `spec.selector`, a NetworkPolicy's `podSelector`.

The deep point: selectors are **not** foreign keys or named references — they're a *loose, live* query. A Service says "any pod with `app=web`" and Kubernetes re-resolves that every moment: add the label, the pod joins the set; remove it, the pod falls out. **That loose coupling is exactly what makes rolling updates possible** — the Service tracks "whoever matches right now," never individual pods. On our map, selectors are the invisible wiring from the **Workloads** band onto the **Pods** they govern.